# Finish the point cloud denoiser - Colab, single session

Does everything outstanding in one run:

1. resume training from epoch 46 to 60 (~2 h)
2. benchmark PU-Net **and** PC-Net (~1.5 h)
3. write the final comparison table

**Set the runtime first:** Runtime -> Change runtime type -> **T4 GPU**.

Checkpoints and results are written to Google Drive, so a disconnect does not
lose the run. Colab's `/content` is wiped when the session ends.


## 1. Drive

Mount so checkpoints survive a disconnect. Everything lands in
`MyDrive/pointdenoise/`.


In [ ]:
import os

USE_DRIVE = True   # False keeps everything in /content and loses it on disconnect

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/pointdenoise"
else:
    OUT = "/content/out"

os.makedirs(OUT, exist_ok=True)
os.makedirs(OUT + "/runs", exist_ok=True)
print("outputs ->", OUT)


## 2. Code and data

Pulls both from Kaggle. `kagglehub.login()` will ask for your username and an
API token from kaggle.com/settings -> Create New Token.


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "kagglehub", "trimesh", "rtree"], check=False)

import kagglehub
kagglehub.login()

CODE_ROOT = kagglehub.dataset_download("yugverma11/pointdenoise-code")
DATA_ROOT = kagglehub.dataset_download("yugverma11/pointdenoise-data")
print("code:", CODE_ROOT)
print("data:", DATA_ROOT)


In [ ]:
def find_containing(*markers, roots=()):
    """First directory under any root that holds one of `markers` as a child."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        for base, dirs, files in os.walk(root):
            for m in markers:
                if m in dirs or m in files:
                    return base
    return None

ROOTS = [CODE_ROOT, DATA_ROOT, OUT, "/content", "/root/.cache/kagglehub"]
CODE = find_containing("pointdenoise", roots=ROOTS)
DATA = find_containing("examples", "PUNet", "PCNet", roots=ROOTS)

assert CODE, "pointdenoise package not found under " + str(ROOTS)
assert DATA, "benchmark data not found under " + str(ROOTS)
sys.path.insert(0, CODE)
print("package:", CODE)
print("dataset:", DATA)

import torch
print()
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NO GPU - Runtime > Change runtime type > T4")
assert torch.cuda.is_available(), "turn the GPU on before continuing"


## 3. Checkpoint

Resuming from epoch 46 costs ~2 h. Starting over costs ~8 h. Put `best.pt`
in `MyDrive/pointdenoise/` beforehand, or use the upload cell below.


In [ ]:
CKPT = None
for base, _, files in os.walk(OUT):
    for f in ("best.pt", "last.pt"):
        if f in files:
            CKPT = os.path.join(base, f)
            break
    if CKPT:
        break

if CKPT is None:
    for base, _, files in os.walk("/content"):
        if "drive" in base:
            continue
        if "best.pt" in files:
            CKPT = os.path.join(base, "best.pt")
            break

print("checkpoint:", CKPT or "NONE - run the upload cell, or training starts from scratch")
if CKPT:
    ck = torch.load(CKPT, map_location="cpu", weights_only=False)
    print(f"  epoch {ck['epoch']}, best loss {ck.get('best'):.6f}, kwargs {ck.get('model_kwargs')}")


In [ ]:
# Only if the cell above found nothing. Uploads best.pt from your machine.
if CKPT is None:
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    CKPT = OUT + "/best.pt"
    with open(CKPT, "wb") as f:
        f.write(up[name])
    ck = torch.load(CKPT, map_location="cpu", weights_only=False)
    print(f"uploaded -> {CKPT}, epoch {ck['epoch']}, best loss {ck.get('best'):.6f}")
else:
    print("checkpoint already found, skipping upload")


## 4. Sanity check before spending GPU hours

Confirms the harness reproduces a published number. If CD calibration fails the
results would be comparable to nothing, so this asserts rather than warns.

P2M is expected to fail: our implementation lands at 0.17x the published value
for the same algorithm, so it measures something else and is never quoted.


In [ ]:
from pointdenoise.benchmark import calibrate, load_released_set

case = load_released_set(DATA, "PUNet", "sparse", 0.01)
print(f"{case.label}: {len(case.shapes)} shapes")
r = calibrate(case)
for m in ("cd", "p2m"):
    print(f"  {m.upper():<4} ours {r['measured_' + m]:7.3f}  published {r['expected_' + m]:6.2f}"
          f"  ratio {r[m + '_ratio']:.2f}x  {'PASS' if r[m + '_ok'] else 'FAIL'}")
print()
print("quotable:", [m.upper() for m in r["comparable_metrics"]])
assert r["cd_ok"], "CD calibration failed - fix before trusting any number"


## 5. Train

Resumes from the checkpoint's epoch. Saves `best.pt` and `last.pt` every epoch
to Drive, so a disconnect costs at most one epoch - just re-run this cell.


In [ ]:
import numpy as np
from pointdenoise.benchmark import load_training_clouds
from pointdenoise.data import Shape
from pointdenoise.engine import train

train_clouds = load_training_clouds(DATA, "PUNet", "sparse")
shapes = [Shape(pts, noise_level=0.02, rng=np.random.default_rng(i))
          for i, (_, pts) in enumerate(train_clouds)]
print(f"{len(shapes)} training shapes, {shapes[0].clean.shape[0]} points each")

EPOCHS = 60
if CKPT:
    done = torch.load(CKPT, map_location="cpu", weights_only=False)["epoch"]
    print(f"resuming at epoch {done}, {EPOCHS - done} to go (~{(EPOCHS - done) * 8 / 60:.1f} h)")

model, history = train(
    shapes,
    out_dir=OUT + "/runs",
    epochs=EPOCHS,
    batch_size=32,
    points_per_patch=256,
    patches_per_shape=1000,
    lr=1e-3,
    repulsion_weight=0.05,
    noise_range=(0.005, 0.03),   # sampled per patch, not fixed
    model_kwargs={"d_model": 256, "num_heads": 8, "num_layers": 6},
    num_workers=2,
    seed=0,
    resume=CKPT,
)


In [ ]:
import matplotlib.pyplot as plt

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4))
a.plot([h["epoch"] for h in history], [h["total"] for h in history], label="total")
a.plot([h["epoch"] for h in history], [h["chamfer"] for h in history], label="chamfer")
a.set_xlabel("epoch"); a.set_ylabel("loss"); a.legend(); a.grid(alpha=.3)
a.set_title("Training loss")
b.plot([h["epoch"] for h in history], [h["lr"] for h in history], color="tab:orange")
b.set_yscale("log"); b.set_xlabel("epoch"); b.set_ylabel("lr"); b.grid(alpha=.3)
b.set_title("Learning rate")
plt.tight_layout(); plt.savefig(OUT + "/loss.png", dpi=120); plt.show()

print(f"epoch {history[0]['epoch']}: {history[0]['total']:.6f}")
print(f"epoch {history[-1]['epoch']}: {history[-1]['total']:.6f}")
print(f"still falling? {history[-1]['total'] < history[-2]['total']}")


## 6. Benchmark both datasets

PU-Net (20 shapes) and PC-Net (10 shapes), each at 10K/50K points and 1/2/3%
noise. The noisy input is scored alongside in every cell, so it is always clear
whether the model helped rather than only how it ranks.


In [ ]:
from pointdenoise.benchmark import NOISE_LEVELS, run_case
from pointdenoise.engine import denoise_cloud

def denoiser(points):
    shape = Shape(np.asarray(points), noisy=np.asarray(points))
    return denoise_cloud(model, shape, points_per_patch=256, batch_size=128, iters=1)

results = {}
for dataset in ("PUNet", "PCNet"):
    scores, baseline = {}, {}
    for resolution in ("sparse", "dense"):
        for noise in NOISE_LEVELS:
            try:
                case = load_released_set(DATA, dataset, resolution, noise)
            except (FileNotFoundError, RuntimeError) as e:
                print(f"skip {dataset}/{resolution}/{noise:.0%}: {e}")
                continue
            _, ours = run_case(case, denoiser, with_p2m=True)
            _, none = run_case(case, lambda p: p, with_p2m=True)
            scores[(resolution, noise)] = ours
            baseline[(resolution, noise)] = none
            gain = (none["cd"] - ours["cd"]) / none["cd"] * 100
            print(f"{dataset}/{resolution}/{noise:.0%}  ours CD {ours['cd']:7.4f}  "
                  f"noisy {none['cd']:7.4f}  {gain:+5.1f}%", flush=True)
    if scores:
        results[dataset] = (scores, baseline)


In [ ]:
from pointdenoise.metrics import paper_table

CAVEAT = (
    "CD is calibrated: this harness reproduces the published Bilateral CD to 0.84x\n"
    "on the same shapes, so the CD columns are comparable.\n\n"
    "P2M is NOT calibrated - 0.17x the published value for the same algorithm - so\n"
    "the P2M columns appear because the layout calls for them, not as a claim.\n"
)

out = []
for dataset, (scores, baseline) in results.items():
    published = None if dataset == "PUNet" else {}
    table = paper_table(scores, our_name="Ours", dataset=dataset, published=published)
    print(table); print()
    out += [table, "", f"{dataset} noisy-input baseline (CD x1e-4)"]
    for k, v in baseline.items():
        o = scores[k]["cd"]
        out.append(f"  {k[0]}/{k[1]:.0%}  ours {o:7.4f}  noisy {v['cd']:7.4f}  "
                   f"{(v['cd'] - o) / v['cd'] * 100:+5.1f}%")
    out.append("")

with open(OUT + "/benchmark.txt", "w") as f:
    f.write("\n".join(out) + "\n\n" + CAVEAT)
print("saved", OUT + "/benchmark.txt")
print("also in Drive:", OUT + "/runs/best.pt", "and", OUT + "/loss.png")


## Done

In `MyDrive/pointdenoise/`:

- `benchmark.txt` - both comparison tables
- `runs/best.pt` - the finished checkpoint
- `runs/history.json` - full loss history
- `loss.png` - training curves

Read the **CD** columns. P2M is not calibrated against the published
definition and is not a claim.
